# Phase 3: Candidate Capability Hunt (Kaggle GPU Runner)

This notebook executes the four rigorous scientific probes on the trained Phase 1 & Phase 2 checkpoints:
1. **Probe 3A: Test-Time Compute-Depth Scaling ($K$-Scaling)**
   - Evaluates Variant D ($d=576$, trained at $K=6$) across additional iterations $K \in \{1, 2, 3, 4, 5, 6, 7, 8, 10, 12\}$.
   - Analyzes compute normalization ($\Delta\text{PPL} / \Delta\text{MFLOP}$) and hard/ambiguous token splits (top 20% next-token entropy).
   - Uses Variant C ($d=288$ unshared) as fixed-depth feedforward control ($K \le 6$).
2. **Probe 3B: Causal Perturbation Attenuation & Recovery Dynamics**
   - Injects controlled impulse perturbation at $t=64$ into intermediate sequence representations.
   - Tracks token-by-token error trajectory $\Delta L_t = L_t^{\text{pert}} - L_t^{\text{clean}}$ across subsequent tokens $t \in [65, 128]$.
   - Measures non-canceling cumulative damage area $D = \sum \max(0, \Delta L_t)$, half-life $t_{1/2}$, and recovery distance $t_{\text{rec}}$.
   - Disentangles cross-factorial controls: Variant D vs Variant C vs Variant A vs Transformer vs GRU.
3. **Probe 3C: Surface Input Noise & Typo Tolerance**
   - Random in-vocabulary token corruption sweep $p \in [0.0..0.20]$.
   - Measures relative degradation ratio $R(p) = \text{PPL}(p) / \text{PPL}(0)$ and degradation slope $\beta$.
4. **Probe 3D: Streaming State Complexity**
   - Measures state memory scaling $M(T) = a + b T$ for incremental autoregressive streaming.
   - Verifies bounded $O(1)$ buffer ($RF=127$) vs Transformer expanding $O(T)$ KV cache.
5. **Decision Gate 3 Evaluation**
   - Rigorous 5-criterion rubric deciding progression to **Phase 4 (Hybrid Adaptor)**.

> **Prerequisite:** Ensure Accelerator is set to **GPU (T4 x1 or P100)** in the right sidebar.

In [ ]:
# Cell 1: Environment Setup & Pull Latest Repository Code
import os, sys
!git clone https://github.com/Zenoguy/NCA-sim.git /kaggle/working/NCA-sim || (cd /kaggle/working/NCA-sim && git pull origin main)
%cd /kaggle/working/NCA-sim
!pip install -q tokenizers pyyaml matplotlib

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Verify Groundwork & Run Unit Test Suite
!pytest tests/ -v

## Execute All Phase 3 Probes on GPU

In [ ]:
# Cell 3: Execute Full Probing Suite & Gate 3 Evaluation
!python scripts/run_level3.py --action all --device cuda

## Generate Publication Figures

In [ ]:
# Cell 4: Plotting Depth Scaling, Perturbation Trajectories, and Robustness Curves
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

out_dir = Path("outputs/level3")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# 1. Plot 3A: Compute-Depth Scaling
ax1 = axes[0, 0]
if (out_dir / "depth_scaling.json").exists():
    with open(out_dir / "depth_scaling.json") as f:
        d_data = json.load(f)
    c_d = d_data.get("variant_d_shared_10m", {}).get("curve", [])
    c_c = d_data.get("variant_c_unshared_10m_control", {}).get("curve", [])
    if c_d:
        ax1.plot([p["K"] for p in c_d], [p["perplexity"] for p in c_d], marker="o", lw=2, label="Variant D (Shared, d=576)", color="#1f77b4")
        if "hard_tokens_top20_ppl" in c_d[0]:
            ax1.plot([p["K"] for p in c_d], [p["hard_tokens_top20_ppl"] for p in c_d], marker="s", ls="--", lw=1.5, label="Variant D (Top 20% Hard)", color="#ff7f0e")
    if c_c:
        ax1.plot([p["K"] for p in c_c], [p["perplexity"] for p in c_c], marker="^", lw=2, label="Variant C (Unshared Control, d=288)", color="#2ca02c")
    ax1.axvline(6, color="red", ls=":", label="Training Depth (K=6)")
    ax1.set_title("Probe 3A: Test-Time Depth Scaling", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Micro-Step Iterations (K)")
    ax1.set_ylabel("Test Perplexity")
    ax1.legend()

# 2. Plot 3B: Perturbation Attenuation Trajectory
ax2 = axes[0, 1]
if (out_dir / "perturbation_attenuation.json").exists():
    with open(out_dir / "perturbation_attenuation.json") as f:
        p_data = json.load(f).get("models", {})
    colors = {"variant_d_shared_10m": "#1f77b4", "variant_c_unshared_10m": "#2ca02c", "variant_a_shared_3m": "#9467bd", "primary_transformer": "#d62728", "gru_baseline": "#8c564b"}
    for k, item in p_data.items():
        traj = item.get("metrics", {}).get("trajectory_subsequent_delta", [])[:32]
        if traj:
            ax2.plot(range(65, 65 + len(traj)), traj, lw=2, label=item["name"], color=colors.get(k, None))
    ax2.set_title("Probe 3B: Causal Perturbation Attenuation Trajectory", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Sequence Token Position (t)")
    ax2.set_ylabel("Error Delta (\Delta L_t)")
    ax2.legend()

# 3. Plot 3C: Surface Noise Relative Degradation
ax3 = axes[1, 0]
if (out_dir / "robustness_relative.json").exists():
    with open(out_dir / "robustness_relative.json") as f:
        r_data = json.load(f).get("models", {})
    for k, item in r_data.items():
        curve = item.get("curve", [])
        if curve:
            ax3.plot([pt["corruption_rate_p"] for pt in curve], [pt["relative_degradation_ratio"] for pt in curve], marker="o", lw=2, label=f"{item['name']} (\beta={item['beta_slope']:.2f})")
    ax3.set_title("Probe 3C: Relative Degradation Under In-Vocab Noise", fontsize=12, fontweight="bold")
    ax3.set_xlabel("Corruption Probability (p)")
    ax3.set_ylabel("Relative Degradation Ratio R(p)")
    ax3.legend()

# 4. Plot 3D: Streaming State Complexity
ax4 = axes[1, 1]
if (out_dir / "streaming_state_complexity.json").exists():
    with open(out_dir / "streaming_state_complexity.json") as f:
        s_data = json.load(f).get("models", {})
    for k, item in s_data.items():
        curve = item.get("curve", [])
        if curve:
            ax4.plot([pt["T"] for pt in curve], [pt["state_memory_mb"] for pt in curve], marker="s", lw=2, label=f"{item['name']} [{item['asymptotic_complexity']}]")
    ax4.set_title("Probe 3D: Streaming State Complexity M(T)", fontsize=12, fontweight="bold")
    ax4.set_xlabel("Generated Sequence Length (T)")
    ax4.set_ylabel("State Memory (MB)")
    ax4.legend()

plt.tight_layout()
plt.savefig(out_dir / "phase3_probes_figure.png", dpi=300)
plt.show()
print(f"Figure saved to: {out_dir / 'phase3_probes_figure.png'}")

## Archive Phase 3 Artifacts for Instant Download

In [ ]:
# Cell 5: Package JSON summaries and figures (tiny, < 1MB, zero .pt model files)
!tar -czvf /kaggle/working/outputs_level3.tar.gz outputs/level3/
print("\nPhase 3 outputs successfully archived to /kaggle/working/outputs_level3.tar.gz (instant download ready!)")